# PUBG Update 42.1 — Steam Review Analysis

## 00. Analysis Question

**How did player reactions and major complaint topics change before and after PUBG Update 42.1?**

This portfolio notebook compares English Steam reviews in two 14-day UTC windows. It checks recommendation and playtime patterns, compares keyword-based issue tags among negative reviews, and separates observed changes from causal claims. The review-level source file is deliberately not published because it contains Steam identifiers and full review text; this notebook reproduces the published aggregation and validation layer from privacy-safe outputs.


## 01. Load Data

The repository contains only aggregate outputs and short, anonymized excerpts. No API call or data collection occurs here.


In [ ]:
from pathlib import Path
import re
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'outputs').is_dir():
    REPO_ROOT = REPO_ROOT.parent
OUTPUTS = REPO_ROOT / 'outputs'

summary = pd.read_csv(OUTPUTS / 'summary_statistics.csv')
validation = pd.read_csv(OUTPUTS / 'data_validation.csv')
issues = pd.read_csv(OUTPUTS / 'negative_issue_comparison.csv')
examples = pd.read_csv(OUTPUTS / 'representative_reviews_public.csv')
assert set(summary['period']) == {'Before', 'After'}
assert set(issues['issue']) == {'ranked_rp', 'matchmaking', 'cheating_fairness', 'performance_technical', 'gameplay_balance'}
print('Loaded privacy-safe portfolio outputs from:', OUTPUTS)


## 02. Data Validation

The completed analysis used the following half-open UTC intervals:

- PRE: `[2026-06-03 08:30, 2026-06-17 08:30)`
- Update proxy: `2026-06-17 08:30`
- POST: `[2026-06-17 08:30, 2026-07-01 08:30)`

The original validation recorded no duplicate recommendation IDs. Blank review text was retained in basic counts but excluded from text analysis.


In [ ]:
validation_view = validation[['period', 'rows', 'duplicate_ids', 'null_review', 'blank_review_including_null', 'oldest_utc', 'newest_utc']]
assert validation.set_index('period').loc['Before', 'rows'] == 667
assert validation.set_index('period').loc['After', 'rows'] == 850
assert validation['duplicate_ids'].eq(0).all()
validation_view


## 03. Basic Comparison

The PRE and POST windows are separate cross-sectional review samples, not the same-player cohort. Playtime is therefore descriptive rather than evidence that the update changed playtime.


In [ ]:
basic = summary.loc[summary['recommendation_group'].eq('All'), [
    'period', 'review_count', 'recommended_count', 'not_recommended_count',
    'recommendation_rate', 'playtime_median_hours', 'review_length_median',
    'helpful_vote_median'
]].copy()
basic['recommendation_rate'] = basic['recommendation_rate'].mul(100).round(2)
basic['playtime_median_hours'] = basic['playtime_median_hours'].round(2)
basic = basic.rename(columns={'recommendation_rate': 'recommendation_rate_pct'})
basic


## 04. Issue Tagging

Issue shares are calculated only among negative reviews. A review can receive multiple tags. The dictionary and exact word-boundary logic below are the rules used in the completed analysis. It is retained for methodological transparency; the private review-level input is not distributed in this repository.


In [ ]:
issue_keywords = {
    'ranked_rp': ['rank', 'ranked', 'rp', 'rating', 'tier', 'point', 'points'],
    'matchmaking': ['matchmaking', 'match making', 'queue', 'teammate', 'teammates'],
    'cheating_fairness': ['cheater', 'cheaters', 'cheating', 'cheat', 'hack', 'hacker', 'hackers'],
    'performance_technical': ['lag', 'fps', 'crash', 'crashing', 'server', 'servers', 'performance', 'optimization'],
    'gameplay_balance': ['balance', 'balanced', 'weapon', 'weapons', 'gun', 'guns', 'damage', 'gameplay'],
}

def tag_negative_reviews(reviews: pd.DataFrame) -> pd.DataFrame:
    # Apply the original multi-label, word-boundary tagging rules.
    tagged = reviews.copy()
    for issue, keywords in issue_keywords.items():
        pattern = r'|'.join(r'\b' + r'\s+'.join(map(re.escape, word.split())) + r'\b' for word in keywords)
        tagged[issue] = tagged['review'].fillna('').str.contains(pattern, case=False, regex=True)
    return tagged


## 05. PRE vs POST Issue Comparison

`change_pp` is POST share minus PRE share in percentage points—not a relative percentage change. The denominator is all negative reviews in each period (PRE n=143; POST n=184).


In [ ]:
issue_view = issues[['issue', 'Before_count', 'Before_negative_denominator', 'Before_mention_pct',
                     'After_count', 'After_negative_denominator', 'After_mention_pct', 'change_pp']].copy()
for column in ['Before_mention_pct', 'After_mention_pct', 'change_pp']:
    issue_view[column] = issue_view[column].round(2)
issue_view


## 06. Context Validation

Keyword detection creates candidates; it does not prove that a review concerns the Update 42.1 RP calculation change. The public excerpts below are deliberately short and remove Steam identifiers. In the completed semantic check, all 16 Ranked/RP keyword candidates were insufficient to validate direct feedback on RP calculation changes (PRE 0/143; POST 0/184).


In [ ]:
examples


## 07. Conclusion

- Recommendation rate was nearly unchanged: **78.56% PRE → 78.35% POST** (-0.21pp).
- Among negative reviews, shares for Ranked/RP, matchmaking, and cheating/fairness declined; performance/technical mentions rose slightly.
- Median playtime at review was higher in POST (50.37h → 89.23h), but this is not a same-user comparison.
- Ranked/RP keyword share declined (6.29% → 3.80%), yet available review context did not support directly attributing that change to the RP calculation update.

These are observed associations in Steam review samples, not evidence of a causal patch effect.
